# 01 · Load, clean and quality-check the sepsis data
**BT3041 Sepsis project** · covers Steps 1–2 of the instruction manual

**What this notebook does:** turns the three raw CSV files into clean tables, checks every sample for quality problems, and writes down each decision so it can go straight into the report's Methods.

| | Files | Where |
|---|---|---|
| **Inputs** | `Genes.csv`, `Genes(479).csv`, `Patients.csv` | `BT3041_Sepsis/data/raw/` |
| **Outputs** | `samples_clean.csv`, `genes_x_samples.csv`, `genes_x_sepsis479.csv`, `genes_x_sepsis479_expressed.csv`, `cleaning_log.csv` | `BT3041_Sepsis/data/processed/` |
| **Figure** | `01_qc.png` | `BT3041_Sepsis/figures/` |

**How to run:** Runtime → Run all (about 2 minutes). Each step prints what it decided. The *Expected* line under each step shows what you should see; tiny differences in the last decimal are fine.

The output files keep the names the manual uses, so Steps 3–11 of the manual run on them unchanged.

In [ ]:
# ---- SETUP CELL: run this first ----
import os
try:
    from google.colab import drive              # running in Google Colab
    drive.mount('/content/drive')               # click the link, allow access
    BASE = "/content/drive/MyDrive/BT3041_Sepsis"
except ImportError:                              # running in Jupyter on a laptop
    BASE = os.environ.get("BT3041_BASE", os.path.abspath(".."))

RAW  = os.path.join(BASE, "data", "raw")
PROC = os.path.join(BASE, "data", "processed")
FIG  = os.path.join(BASE, "figures")
RES  = os.path.join(BASE, "results")
for d in [PROC, FIG, RES]:
    os.makedirs(d, exist_ok=True)
print("BASE:", BASE)
print("raw files:", sorted(os.listdir(RAW)))

Expected: the last line lists `['Genes(479).csv', 'Genes.csv', 'Patients.csv', ...]`.
If you get `FileNotFoundError`, the shared folder shortcut is missing from *My Drive*, or the files are not in `data/raw/`.

## 1. Load the three files
- `Genes.csv`: one row per gene, one column per blood sample (802). Values are **log2** expression (RMA-normalised by the original authors).
- `Genes(479).csv`: the same genes, only the 479 sepsis patients.
- `Patients.csv`: one row per sample: age, sex, group, 28-day outcome, endotype and other labels.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

genes = pd.read_csv(os.path.join(RAW, "Genes.csv"), index_col=0)        # genes x 802 samples
g479  = pd.read_csv(os.path.join(RAW, "Genes(479).csv"), index_col=0)   # genes x 479 sepsis samples
pat   = pd.read_csv(os.path.join(RAW, "Patients.csv"), index_col=0)     # 802 samples x 15 columns

print("Genes.csv      ", genes.shape)
print("Genes(479).csv ", g479.shape)
print("Patients.csv   ", pat.shape)
pat.head(3)

Expected: `(11519, 802)`, `(11519, 479)`, `(802, 15)` (takes ~20 s).

### A decisions log
Every cleaning choice is recorded with `note(...)`. At the end the log is saved as `cleaning_log.csv`; that file is your Methods paragraph in table form.

In [ ]:
log = []
def note(step, decision, detail):
    """Record one cleaning decision and print it."""
    log.append({"step": step, "decision": decision, "detail": detail})
    print(f"[{step}] {decision}: {detail}")

## 2. Clean the gene table
- **`---` row:** Affymetrix writes `---` for probe sets that do not map to any gene. It is not a gene, so we drop it.
- **Missing values / duplicates:** checked; there are none.
- **`Genes(479).csv`:** we check whether it is just the sepsis columns of `Genes.csv`. If it is, we don't need it; we rebuild it from the cleaned full table so both always match.

In [ ]:
n0 = len(genes)
genes = genes.drop(index=[g for g in genes.index if g == "---"])
note("genes", "dropped '---' row", f"{n0} -> {len(genes)} genes (probe sets without a gene symbol)")

assert genes.isna().sum().sum() == 0 and not genes.index.duplicated().any()
note("genes", "no missing values / duplicate genes",
     f"range {genes.values.min():.2f}-{genes.values.max():.2f} (already log2 RMA)")

same = np.allclose(genes[g479.columns].values, g479.drop(index="---").values)
note("genes", "Genes(479).csv is a column subset of Genes.csv",
     f"identical values: {same}; regenerated from the cleaned full table")

Expected: `11519 -> 11518 genes`, range `1.18-13.54`, `identical values: True`.

Values between ~1 and ~14 confirm the data are already on the log2 scale (raw intensities would reach tens of thousands).

## 3. Clean the patient table
| Change | Why |
|---|---|
| `Sepsis_with_outcome` → `Sepsis`, `sample_title` → `title` | Names the manual's code expects |
| Drop `source` | Same value (`whole blood`) for everyone, so it carries no information |
| `male` (0/1), `diabetes` (0/1) | Numbers that regression models can use directly |
| Standardise `icu_acquired_infection_paired` | Mixed spelling: `Admis_ICUAI` vs `ADMIS_NO_ICUAI` |
| `thrombocytopenia_score` 0–3 | The letter prefix (A very low … D normal) is an order, so we keep it as a number |
| `infection_source` | One column instead of two half-empty ones (pneumonia / abdominal) |
| `run_date`, `array_well` | Hidden in the sample title, e.g. `[03_04_2013_B02_3270]` = 3 April 2013, well B02. Run date is our **batch** variable |
| `hospital` | `discovery` = Amsterdam, `validation` = Utrecht |

In [ ]:
pat = pat.rename(columns={"sample_title": "title"})
pat["group"] = pat["group"].replace({"Sepsis_with_outcome": "Sepsis"})
note("patients", "renamed group 'Sepsis_with_outcome' -> 'Sepsis', 'sample_title' -> 'title'", "matches the manual's code")

pat = pat.drop(columns=["source"])
note("patients", "dropped 'source'", "constant ('whole blood' for all 802)")

pat["gender"] = pat["gender"].str.lower().str.strip()
pat["male"] = (pat["gender"] == "male").astype(int)

# outcome as whole numbers; only defined for sepsis patients
pat["mortality_event_28days"] = pat["mortality_event_28days"].astype("Int64")
pat["time_to_event_28days"] = pat["time_to_event_28days"].astype("Int64")

pat["icu_acquired_infection_paired"] = pat["icu_acquired_infection_paired"].replace(
    {"Admis_ICUAI": "admission_ICUAI", "ADMIS_NO_ICUAI": "admission_no_ICUAI", "Event_ICUAI": "event_ICUAI"})
note("patients", "standardised icu_acquired_infection_paired spelling", "Admis_/ADMIS_ mixed case")

pat["thrombocytopenia_score"] = pat["thrombocytopenia"].map(
    {"A_very_low": 0, "B_medium_low": 1, "C_low": 2, "D_normal": 3}).astype("Int64")
pat["diabetes"] = pat["diabetes_mellitus"].map({"DM": 1, "No_DM": 0}).astype("Int64")

# one infection-source column for sepsis patients
src = pd.Series("other/unknown", index=pat.index)
src[pat.pneumonia_diagnoses == "cap"] = "CAP"
src[pat.pneumonia_diagnoses == "hap"] = "HAP"
src[pat.abdominal_sepsis_and_controls == "abdo_s"] = "abdominal"
both = pat.pneumonia_diagnoses.isin(["cap", "hap"]) & (pat.abdominal_sepsis_and_controls == "abdo_s")
src[both] = "pneumonia+abdominal"
pat["infection_source"] = src.where(pat.group == "Sepsis")
note("patients", "derived infection_source for sepsis",
     f"{pat.infection_source.value_counts().to_dict()}; both labels: {pat.index[both & (pat.group == 'Sepsis')].tolist()}")

# batch information hidden in the sample title (dates are day_month_year)
pat["run_date"] = pd.to_datetime(pat["title"].str.extract(r"\[(\d\d_\d\d_\d{4})")[0], format="%d_%m_%Y")
pat["array_well"] = pat["title"].str.extract(r"\[\d\d_\d\d_\d{4}_([A-H]\d\d)")[0]
note("patients", "extracted run_date and array_well from title", f"{pat.run_date.nunique()} run dates (batch variable)")

pat["hospital"] = pat["endotype_cohort"].map({"discovery": "Amsterdam", "validation": "Utrecht"})
print("\ngroups:", pat.group.value_counts().to_dict())

Expected: infection source `other/unknown 248, CAP 106, HAP 76, abdominal 48, pneumonia+abdominal 1`; `14 run dates`; groups `Sepsis 479, Other_ICU 281, Healthy 42`.

## 4. Check the outcome columns
For sepsis patients we have `mortality_event_28days` (1 = died within 28 days) and `time_to_event_28days` (day of death, or 28 if alive). Four kinds of rows look odd at first:

| Rows | What they mean | Decision |
|---|---|---|
| Died at **day 0** (11) | Died on the day of ICU admission | Keep: valid |
| Died at **day 28** (2) | Died on the last day of follow-up | Keep: still "within 28 days" |
| Survived but time **< 28** (2) | Followed only 5 or 8 days (e.g. transferred) = **censored early**; we don't know if they survived to day 28 | Keep, **flag**; drop them in a sensitivity check |
| Age **17** (1) | Plausible ICU admission | Keep |

We do **not** fill in (impute) missing diabetes: guessing a diagnosis would be making data up. Models that use diabetes simply drop those patients (complete-case analysis) and we say so.

In [ ]:
sep = pat.group == "Sepsis"
died = pat.mortality_event_28days == 1
t = pat.time_to_event_28days
pat["flag_censored_early"] = (sep & (pat.mortality_event_28days == 0) & (t < 28)).fillna(False).astype(bool)
pat["flag_death_day0"] = (sep & died & (t == 0)).fillna(False).astype(bool)

note("outcome", "kept, flagged 2 survivors followed < 28 days",
     f"{pat.index[pat.flag_censored_early].tolist()} (censored at day {t[pat.flag_censored_early].tolist()}); "
     f"their 'survived' label is uncertain -> sensitivity analysis")
note("outcome", "kept deaths at day 0", f"{int(pat.flag_death_day0.sum())} deaths on the admission day; valid for Kaplan-Meier/Cox")
note("outcome", "kept deaths at day 28", f"{int((sep & died & (t == 28)).sum())}; death on day 28 is still within 28 days")
note("outcome", "outcome missing only outside sepsis",
     f"sepsis missing outcome = {pat.loc[sep, 'mortality_event_28days'].isna().sum()}")
note("patients", "kept age < 18", f"{pat.index[pat.age < 18].tolist()}; plausible adult-ICU admission, ages {pat.age.min()}-{pat.age.max()}")

dm_missing = pat.loc[sep, "diabetes"].isna().sum()
note("patients", "diabetes left missing (no imputation)",
     f"{dm_missing}/479 sepsis patients ({100*dm_missing/479:.1f}%) -> complete-case in models that use it")
print("\nmissing values per column (all 802 samples):")
print(pat.isna().sum()[pat.isna().sum() > 0].to_string())

Expected: censored early `GSM1602888` (day 8) and `GSM1692379` (day 5); 11 deaths at day 0; 2 at day 28; 0 sepsis patients missing an outcome; diabetes missing for 89 (18.6%).

Most other missing values are **by design**: e.g. healthy controls have no endotype or outcome, and only sub-studies have pneumonia or platelet labels.

## 5. Sample quality control
### 5a. Arrays that look unusual
We correlate every sample with the "median sample". A failed array would correlate much worse than the rest. Samples more than 4 SD below the average are flagged.

In [ ]:
X = genes[pat.index]
corr_med = X.corrwith(X.median(axis=1))
z = (corr_med - corr_med.mean()) / corr_med.std()
pat["qc_corr_to_median"] = corr_med.round(4)
pat["flag_expr_outlier"] = z < -4
note("QC", "kept all arrays; flagged z<-4 outliers",
     f"{int(pat.flag_expr_outlier.sum())} flagged (corr to median sample {corr_med.min():.3f}-{corr_med.max():.3f}); "
     f"groups: {pat.loc[pat.flag_expr_outlier, 'group'].value_counts().to_dict()}")
pat.loc[pat.flag_expr_outlier, ["group", "endotype_class", "mortality_event_28days", "qc_corr_to_median"]]

Expected: 9 flagged (5 sepsis, 4 other ICU), correlations 0.756–0.985.

**Decision: keep them.** A correlation above 0.75 is still high, so the arrays did not fail. Most flagged sepsis samples are **Mars1**, the sickest endotype, so their unusual profile is probably real biology. Removing them would bias us against finding Mars1.

### 5b. Sex check: does the blood agree with the recorded sex?
Two genes act as a built-in ID check:
- **XIST** is switched on only in females (it silences the second X chromosome).
- **RPS4Y1** sits on the Y chromosome, so only males express it.

Each gene's values split into two clear groups with an empty gap between them (XIST 3.0–4.0, RPS4Y1 4.5–6.0), so we cut inside the gaps. A sample whose genes disagree with its recorded sex is a likely **sample or label mix-up**. If both genes are on, the tube may contain blood from two people ("mixed").

In [ ]:
y_on = X.loc["RPS4Y1"] > 5.0
x_on = X.loc["XIST"] > 3.5
pat["expr_sex"] = np.select([y_on & ~x_on, x_on & ~y_on, y_on & x_on], ["male", "female", "mixed"], "unclear")
pat["flag_sex_mismatch"] = pat.expr_sex != pat.gender
note("QC", "sex check (RPS4Y1 vs XIST)",
     f"{int(pat.flag_sex_mismatch.sum())} samples where expression sex != recorded sex "
     f"({pat.loc[pat.flag_sex_mismatch, 'expr_sex'].value_counts().to_dict()}); "
     f"groups {pat.loc[pat.flag_sex_mismatch, 'group'].value_counts().to_dict()}. Kept, flagged -> sensitivity analysis")
pat.loc[pat.flag_sex_mismatch, ["gender", "expr_sex", "group", "age", "hospital", "mortality_event_28days", "run_date"]]

Expected: 8 samples (3 look female, 3 look male, 2 mixed; 5 sepsis, 3 other ICU).

**Pattern:** they come in pairs run on the **same day**. `GSM1602833` (recorded female, looks male) and `GSM1691979` (recorded male, looks female) were both run on 6 Feb 2013, which looks like a direct swap. The other pairs share 5 Mar and 14 Mar.

**Decision: keep and flag.** If two samples were swapped, their age and outcome may belong to someone else, so we re-run key results without them.

### 5c. Duplicate arrays?
If the same patient were sampled twice, two arrays would be almost identical. We correlate every pair of samples on the 2,000 most variable genes. Each group is centred on its own average first; otherwise all healthy controls look alike simply because they are all healthy.

In [ ]:
top = X.var(axis=1).sort_values(ascending=False).index[:2000]
Xc = X.loc[top].copy()
for _, ids in pat.groupby("group").groups.items():
    Xc[ids] = Xc[ids].sub(Xc[ids].mean(axis=1), axis=0)
C = np.corrcoef(Xc.T.values)
np.fill_diagonal(C, np.nan)
iu = np.triu_indices_from(C, 1)
pairs = (pd.DataFrame({"a": X.columns[iu[0]], "b": X.columns[iu[1]], "r": C[iu]})
         .sort_values("r", ascending=False).reset_index(drop=True))
for c in ["a", "b"]:
    for col in ["group", "age", "gender"]:
        pairs[f"{c}_{col}"] = pat.loc[pairs[c], col].values
pat["max_corr_other_sample"] = np.nanmax(C, axis=1).round(3)
note("QC", "no duplicate arrays", f"pairs with r > 0.9: {(pairs.r > 0.9).sum()}; highest r = {pairs.r.iloc[0]:.3f}, "
     f"and the top pairs differ in age/sex, so they are different people")
pairs.to_csv(os.path.join(RES, "qc_most_similar_pairs.csv"), index=False)
pairs.head(5).round(3)

Expected: no pair above 0.9; highest r = 0.870, between people of different ages. **No duplicates.**

### 5d. QC figure (for the report / slide 3)

In [ ]:
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
PAL = {"Healthy": "#2a9d8f", "Other_ICU": "#8d99ae", "Sepsis": "#e63946"}
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))

for g, col in PAL.items():                                   # A: distributions
    for s in pat.index[pat.group == g][:25]:
        sns.kdeplot(genes[s], ax=ax[0], color=col, lw=0.6, alpha=0.5)
ax[0].set(title="A  Expression density (25 samples/group)", xlabel="log2 expression")
ax[0].legend(handles=[plt.Line2D([], [], color=c, label=g) for g, c in PAL.items()], frameon=False)

o = pat.sort_values("qc_corr_to_median")                     # B: outliers
ax[1].scatter(range(len(o)), o.qc_corr_to_median, s=6, c=np.where(o.flag_expr_outlier, "#e63946", "#8d99ae"))
ax[1].set(title=f"B  Correlation with median sample ({int(pat.flag_expr_outlier.sum())} flagged, kept)",
          xlabel="samples (sorted)", ylabel="Pearson r")

gx = genes.loc[["XIST", "RPS4Y1"]].T.join(pat[["gender", "flag_sex_mismatch"]])   # C: sex check
for sx, col in [("female", "#e76f51"), ("male", "#264653")]:
    d = gx[gx.gender == sx]
    ax[2].scatter(d.XIST, d.RPS4Y1, s=8, c=col, alpha=0.6, label=f"recorded {sx}")
mm = gx[gx.flag_sex_mismatch]
ax[2].scatter(mm.XIST, mm.RPS4Y1, s=70, facecolors="none", edgecolors="red", lw=1.5, label=f"mismatch (n={len(mm)})")
ax[2].axvline(3.5, ls=":", c="grey"); ax[2].axhline(5, ls=":", c="grey")
ax[2].set(title="C  Sex check: XIST (X) vs RPS4Y1 (Y)", xlabel="XIST", ylabel="RPS4Y1")
ax[2].legend(frameon=False, fontsize=8)

plt.tight_layout(); plt.savefig(os.path.join(FIG, "01_qc.png"), dpi=200); plt.show()

**What you should see:** (A) curves almost on top of each other, so the normalisation worked; (B) a smooth curve with a few red points at the low end; (C) two tight clouds (males top-left, females bottom-right) with 8 red circles in the "wrong" cloud or in between.

## 6. Low-expression filter
Genes with an average below 3 (log2) are essentially switched off in blood. They add noise and extra tests, so we drop them **for gene-by-gene testing and prediction only** (Steps 7, 8, 11). The full table is kept for PCA and clustering.

In [ ]:
sep_ids = pat.index[sep]
mean_sep = genes[sep_ids].mean(axis=1)
expressed = mean_sep[mean_sep >= 3].index
note("genes", "expressed-gene set for DE/ML",
     f"mean >= 3 in sepsis: {len(expressed)} of {len(genes)} genes kept (full table kept for PCA/clustering)")

Expected: `8200 of 11518`. (The manual says 8,201; the difference is the `---` row we removed.)

## 7. Save everything
From now on, every notebook starts from these files, never from the raw ones.

In [ ]:
order = ["title", "group", "gender", "male", "age", "hospital", "endotype_cohort", "endotype_class",
         "mortality_event_28days", "time_to_event_28days", "infection_source", "pneumonia_diagnoses",
         "abdominal_sepsis_and_controls", "diabetes_mellitus", "diabetes", "thrombocytopenia",
         "thrombocytopenia_score", "icu_acquired_infection", "icu_acquired_infection_paired",
         "run_date", "array_well", "flag_censored_early", "flag_death_day0", "flag_expr_outlier",
         "flag_sex_mismatch", "expr_sex", "qc_corr_to_median", "max_corr_other_sample"]
pat = pat[order]
pat.to_csv(os.path.join(PROC, "samples_clean.csv"))
genes.round(5).to_csv(os.path.join(PROC, "genes_x_samples.csv"))
genes[sep_ids].round(5).to_csv(os.path.join(PROC, "genes_x_sepsis479.csv"))
genes.loc[expressed, sep_ids].round(5).to_csv(os.path.join(PROC, "genes_x_sepsis479_expressed.csv"))
pd.DataFrame(log).to_csv(os.path.join(PROC, "cleaning_log.csv"), index=False)

print("saved:", sorted(os.listdir(PROC)))
print("groups:", pat.group.value_counts().to_dict(),
      "| sepsis deaths:", int(pat.loc[sep, "mortality_event_28days"].sum()), "of", int(sep.sum()))
pd.DataFrame(log)

Expected: 5 files saved (~150 MB, takes ~30 s on Drive); `Sepsis 479, Other_ICU 281, Healthy 42`; `114 of 479` deaths.

## Summary for the report (Methods → Data preprocessing)
| Decision | Choice | Why |
|---|---|---|
| Normalisation | Use the authors' RMA log2 values | Already normalised; distributions match (Fig A) |
| Unannotated probes | Dropped `---` row | Not a gene |
| Outlier arrays | Keep all 802; 9 flagged | Correlation ≥ 0.75; mostly Mars1, i.e. biology |
| Sex mismatches | Keep; 8 flagged | Likely label swaps; sensitivity analysis without them |
| Early-censored survivors | Keep; 2 flagged | Outcome uncertain; sensitivity analysis |
| Duplicates | None found | Max between-sample r = 0.87, different people |
| Missing diabetes | Not imputed | Complete-case analysis (n = 390) |
| Batch | Run date extracted (14 dates) | Tested in notebook 02, not corrected |
| Low expression | Mean < 3 removed for DE/ML only | 8,200 expressed genes |

**Next:** run `02_explore_patterns.ipynb`.